# Two-Target (canoe)

**What you will learn:** solve sub-problems separately, combine tubes with
`union` / `intersect`, run `reach` on the result, and extract a trajectory with
closed-loop trajectory extraction.

**pyspect API:** `TVHJImpl`, `reach`, `union`, `intersect`, `make_tube`


A canoe must visit **both** targets in either order before the horizon ends. A
state-dependent current pushes it left.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt


from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Canoe

In [ ]:
# Same grid as hjr_examples: x, y in [-10,10], horizon 17.5 s
AXES = [
    dict(name='t', bounds=[0, 17.5], points=100),
    dict(name='x', bounds=[-10, 10], points=100),
    dict(name='y', bounds=[-10, 10], points=100),
]

impl = TVHJImpl(dict(cls=Canoe), AXES, accuracy='very_high')

S = impl.grid.states
l1 = jnp.linalg.norm(S[..., [0, 1]] - jnp.array([5.0, 5.0]), axis=-1) - 1.0
l2 = jnp.linalg.norm(S[..., [0, 1]] - jnp.array([-5.0, 5.0]), axis=-1) - 1.0

In [ ]:
# The current: zero for x >= 0, then of magnitude x/4 towards the left
fig, ax = plt.subplots(figsize=(6, 6))
th = np.linspace(-np.pi, np.pi, 100)
ax.plot(5 + np.cos(th), 5 + np.sin(th), color='g', label='Targets')
ax.plot(-5 + np.cos(th), 5 + np.sin(th), color='g')

gx, gy = np.meshgrid(np.linspace(-8, 8, 12), np.linspace(-8, 8, 12))
ax.quiver(gx, gy, np.minimum(gx / 4.0, 0.0), 0.0 * gy, color='b',
          label='Disturbance bound')

ax.set_xlim([-10, 10]); ax.set_ylim([-10, 10])
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('The two-target problem')
ax.legend(framealpha=1.0)
plt.show()

## Step 1 - each target on its own


In [ ]:
V1 = impl.reach(l1)   # tube towards the right-hand target
V2 = impl.reach(l2)   # tube towards the left-hand target

ttr = np.array(impl.timeline)[::-1]   # time-to-go associated with each index
print('V1', V1.shape, '| coverage at full horizon:', f'{100*float((V1[0] <= 0).mean()):.1f} %')
print('V2', V2.shape, '| coverage at full horizon:', f'{100*float((V2[0] <= 0).mean()):.1f} %')

In [ ]:
EXT = [-10, 10, -10, 10]
X = np.array(S[..., 0]); Y = np.array(S[..., 1])

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, W, name in [(axs[0], V1, 'right target (5, 5)'), (axs[1], V2, 'left target (-5, 5)')]:
    im_ = ax.imshow(np.array(W[0]).T, origin='lower', extent=EXT, aspect='equal', cmap='viridis')
    ax.contour(X.T, Y.T, np.array(W[0]).T, levels=[0], colors='black', linewidths=1.4)
    ax.contour(X.T, Y.T, np.array(l1).T, levels=[0], colors='lime', linewidths=1.2)
    ax.contour(X.T, Y.T, np.array(l2).T, levels=[0], colors='lime', linewidths=1.2)
    plt.colorbar(im_, ax=ax)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f'BRT, {name}')
plt.tight_layout(); plt.show()

## Step 2 - compose the two

"Visit both targets" splits into two scenarios: either you hit the left target while still
being able to reach the right one, or the other way round. As sets:

```
inner = (left_target and right_BRT) or (right_target and left_BRT)
```

`intersect` and `union` in `TVHJImpl` are exactly the `max` and `min` of the value
functions. All that is left is to run `reach` on that intermediate set, which is this time
time-varying.


In [ ]:
inner = impl.union(
    impl.intersect(impl.make_tube(l2), V1),
    impl.intersect(impl.make_tube(l1), V2),
)

V = impl.reach(inner)
print('V', V.shape, '| coverage at full horizon:', f'{100*float((V[0] <= 0).mean()):.1f} %')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im_ = ax.imshow(np.array(V[0]).T, origin='lower', extent=EXT, aspect='equal', cmap='viridis')
ax.contour(X.T, Y.T, np.array(V[0]).T, levels=[0], colors='black', linewidths=1.6)
ax.contour(X.T, Y.T, np.array(l1).T, levels=[0], colors='lime', linewidths=1.4)
ax.contour(X.T, Y.T, np.array(l2).T, levels=[0], colors='lime', linewidths=1.4)
plt.colorbar(im_, ax=ax, label=r'$V(x,t)$')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Where can both targets be visited within {ttr[0]:.1f} s?')
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

V_np = np.array(V)
vmin, vmax = float(V_np.min()), float(V_np.max())
fig, ax = plt.subplots(figsize=(6, 5))

def update(i):
    ax.clear()
    ax.imshow(V_np[i].T, origin='lower', extent=EXT, aspect='equal',
              cmap='viridis', vmin=vmin, vmax=vmax)
    ax.contour(X.T, Y.T, V_np[i].T, levels=[0], colors='black', linewidths=1.4)
    ax.contour(X.T, Y.T, np.array(l1).T, levels=[0], colors='lime', linewidths=1.0)
    ax.contour(X.T, Y.T, np.array(l2).T, levels=[0], colors='lime', linewidths=1.0)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'time-to-go = {ttr[i]:.1f} s')

frames = list(range(len(V_np) - 1, -1, -1))
ani = FuncAnimation(fig, update, frames=frames, interval=200, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


## Step 3 - extract a trajectory

The value function does not only say *whether* it is feasible: its gradient gives the
optimal control at every instant. So we integrate the closed-loop dynamics, switching to
the tube of the remaining target as soon as the first one is reached.


In [ ]:
from scipy.integrate import solve_ivp

tl = np.array(impl.timeline)
T = float(tl[-1])
dyn = impl.reach_dynamics

def closed_loop_rhs(impl, dyn, W, timeline):
    """Closed-loop velocity field guided by value-function tube W."""
    W_np = np.asarray(W)
    tl = np.asarray(timeline)
    T = float(tl[-1])
    Wt = jnp.flip(jnp.asarray(W_np), axis=0)          # index by time-to-go
    grads = [impl.grid.grad_values(Wt[j]) for j in range(len(tl))]

    def rhs(t, x):
        j = int(np.abs(tl - max(T - t, 0.0)).argmin())
        xs = jnp.asarray(x)
        g = impl.grid.interpolate(grads[j], state=xs)
        u = dyn.optimal_control(xs, 0.0, g)
        d = dyn.optimal_disturbance(xs, 0.0, g)
        return np.array(dyn.open_loop_dynamics(xs, 0.0)
                        + dyn.control_jacobian(xs, 0.0) @ u
                        + dyn.disturbance_jacobian(xs, 0.0) @ d)
    return rhs

def value_at(impl, level_set, x):
    return float(impl.grid.interpolate(jnp.asarray(level_set), state=jnp.asarray(x)))

In [ ]:
x0 = [0.0, 0.0]
sol = solve_ivp(closed_loop_rhs(impl, dyn, V, tl), [0, T], x0, max_step=0.1)

d1 = np.array([value_at(impl, l1, sol.y[:, i]) for i in range(len(sol.t))])
d2 = np.array([value_at(impl, l2, sol.y[:, i]) for i in range(len(sol.t))])
i1 = int(np.argmax(d1 < -0.5)) if (d1 < -0.5).any() else len(sol.t)
i2 = int(np.argmax(d2 < -0.5)) if (d2 < -0.5).any() else len(sol.t)

k = min(i1, i2)
first, remaining = ('right', V2) if i1 < i2 else ('left', V1)
print(f'first target reached: {first} at t = {sol.t[k]:.2f} s')

sol2 = solve_ivp(closed_loop_rhs(impl, dyn, remaining, tl), [sol.t[k], T], sol.y[:, k], max_step=0.1)
ts = np.concatenate([sol.t[:k], sol2.t])
ys = np.concatenate([sol.y[:, :k], sol2.y], axis=1)
print(f'final position at t = {ts[-1]:.2f} s: ({ys[0, -1]:.2f}, {ys[1, -1]:.2f})')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(ys[0, :], ys[1, :], 'k-', lw=2, label='Trajectory')
ax.plot(*x0, 'ko', ms=8, label='Start')
ax.plot(ys[0, k], ys[1, k], 'o', color='orange', ms=9, label='First target reached')

th = np.linspace(-np.pi, np.pi, 100)
ax.plot(5 + np.cos(th), 5 + np.sin(th), color='g', label='Targets')
ax.plot(-5 + np.cos(th), 5 + np.sin(th), color='g')
ax.contour(X.T, Y.T, np.array(V[0]).T, levels=[0], colors='gray', linewidths=1.0)

gx, gy = np.meshgrid(np.linspace(-8, 8, 12), np.linspace(-8, 8, 12))
ax.quiver(gx, gy, np.minimum(gx / 4.0, 0.0), 0.0 * gy, color='b', alpha=0.5)

ax.set_xlim([-10, 10]); ax.set_ylim([-10, 10])
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Optimal trajectory visiting both targets')
ax.legend(framealpha=1.0)
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(7, 7))
th = np.linspace(-np.pi, np.pi, 100)
ax.plot(5 + np.cos(th), 5 + np.sin(th), color='g')
ax.plot(-5 + np.cos(th), 5 + np.sin(th), color='g')
ax.contour(X.T, Y.T, np.array(V[0]).T, levels=[0], colors='gray', linewidths=1.0)
gx, gy = np.meshgrid(np.linspace(-8, 8, 12), np.linspace(-8, 8, 12))
ax.quiver(gx, gy, np.minimum(gx / 4.0, 0.0), 0.0 * gy, color='b', alpha=0.5)
ax.set_xlim([-10, 10]); ax.set_ylim([-10, 10])
ax.set_xlabel('x'); ax.set_ylabel('y')

(traj,) = ax.plot([], [], 'k-', lw=2)
(start,) = ax.plot([], [], 'ko', ms=8)
(marker,) = ax.plot([], [], 'o', color='orange', ms=9)

def update(frame):
    n = frame + 1
    traj.set_data(ys[0, :n], ys[1, :n])
    start.set_data([x0[0]], [x0[1]])
    mk = min(n - 1, k)
    marker.set_data([ys[0, mk]], [ys[1, mk]])
    ax.set_title(f'Trajectory, t = {ts[min(n - 1, len(ts) - 1)]:.2f} s')
    return traj, start, marker

ani = FuncAnimation(fig, update, frames=len(ts), interval=80, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())
